# Using cmsopen one layer at a time

`cmsopen.analyze` runs the whole chain. Each step is also usable by itself, so you can change one and keep the rest. This
notebook walks through them on the committed sample, using the `Z_mumu` channel's frozen configuration.

In [ ]:
import json
import os
import re
from pathlib import Path

from IPython.display import Image, Markdown, display

import cmsopen
from cmsopen.fitting import pdg_mass_width
from cmsopen.pipeline import CHANNELS

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "data" / "cmsopen_sample.root").exists())
SAMPLE = ROOT / "examples" / "data" / "cmsopen_sample.root"
FIGURES = ROOT / "docs" / "figures"
PRODUCTION = json.loads((ROOT / "results" / "production.json").read_text())
print(f"cmsopen {cmsopen.__version__} imported from: {cmsopen.__file__}")


class Checks:
    """Collects self-check results and prints one OK/FAIL line each."""

    def __init__(self):
        self.failures = []

    def check(self, name, ok, detail=""):
        """Print one OK or FAIL line and remember failures."""
        print(f"{'OK' if ok else 'FAIL'}: {name}" + (f" ({detail})" if detail else ""))
        if not ok:
            self.failures.append(name)


checks = Checks()

## Read events and select muons

`open_events` opens a file lazily. `events.muons(...)` builds muons with cuts, and `require_trigger` and `require` act on whole events.

In [ ]:
import awkward as ak
import numpy as np

from cmsopen import (
    ZMUMU_MUON_CUTS,
    best_pair,
    fit_mass_peak,
    make_histogram,
    open_events,
    opposite_sign_pairs,
    plot_mass_spectrum,
    same_sign_pairs,
)
from cmsopen.ingestion import DIMUON_TRIGGERS_2012

with open_events(SAMPLE) as events:
    print(events.summary())
    selected = events.muons(**ZMUMU_MUON_CUTS).require_trigger(*DIMUON_TRIGGERS_2012).require(2)
print(selected)

## Form pairs and compute masses

An event can hold more than two selected muons. `best_pair` keeps one opposite-sign pair per event with a rule that does not use
the mass, so the choice cannot sculpt a peak.

In [ ]:
opposite = opposite_sign_pairs(selected)
same = same_sign_pairs(selected)
best = best_pair(opposite, "highest_pt_sum")
masses = ak.to_numpy(ak.flatten(best.mass))
print(f"{best.n_pairs:,} opposite-sign pairs, one per event")
checks.check("at most one pair per event", best.n_pairs <= selected.n_events, f"{selected.n_events:,} events")

## Histogram and fit

The channel's configuration holds the frozen histogram and fit range. The model name is yours to choose. The numbers printed here are a
demonstration on the sample and are not results.

In [ ]:
config = CHANNELS["Z_mumu"]
histogram = make_histogram(masses, bins=config.bins, range=config.mass_range)
fit = fit_mass_peak(histogram, model="voigt_poly", fit_range=config.fit_range)
print(fit.table())
figure, *_ = plot_mass_spectrum(histogram, fit)
display(figure)

## The same fit through the pipeline

`analyze` is these steps with chunking, parallel workers, an alternate fit, and summaries. On this sample both routes must find the same pairs.

In [ ]:
result = cmsopen.analyze(SAMPLE, channel="Z_mumu", model="voigt_poly", stability=False, verbose=False)
same_masses = np.array_equal(np.sort(masses), np.sort(result.masses))
checks.check("the layers and the pipeline find the same masses", same_masses, f"{len(masses):,} masses")
checks.check("the layers and the pipeline give the same peak",
             abs(fit.peak_position[0] - result.fit.peak_position[0]) < 1e-6)
pdg_mass = pdg_mass_width(config.resonance)[0]
checks.check("the peak is within a generous window of the PDG mass", abs(fit.peak_position[0] - pdg_mass) < 1.0)

## Optional: your own file

If `CMSOPEN_DATA_FILE` points to a NanoAOD file, this reads its first events through the same layers.

In [ ]:
data_file = os.environ.get("CMSOPEN_DATA_FILE")
if data_file:
    with open_events(data_file) as events:
        print(events.summary())
else:
    print("CMSOPEN_DATA_FILE is not set: skipped")

## Production results

The same layers, run by `cmsopen.analyze` with the frozen models over the pre-registered sample, gave the values below. Benchmarks are requirements fixed before
the sample was opened; observed values are what was measured.



Measured on 20 files (controlling **benchmark**: 20 pre-registered files, met).

| Z → μ⁺μ⁻ | value |
|---|---|
| Model and fit range | `cb2_bw_exp`, 70-110 GeV |
| Events read, pairs in the fit range | 9,780,165, 584,716 |
| Peak (pole) position, observed | 90.9000 GeV |
| Statistical error, raw | 0.0050 GeV |
| Statistical error, scaled by S = 1.232 | 0.0061 GeV |
| Resolution, observed | 1.9821 ± 0.0075 GeV |
| χ²/ndf **benchmark** | ≤ 2.0 |
| χ²/ndf, observed | 48.6 / 32 = 1.518 (met) |
| PDG mass (reference only) | 91.1879 GeV |
| Offset from PDG, observed | -0.2879 GeV |
| Fit-choice spread, observed | 0.0028 GeV |



Measured on 20 files (controlling **benchmark**: 20 pre-registered files, met).

| J/ψ → μ⁺μ⁻ | value |
|---|---|
| Model and fit range | `cb2_bw_poly2`, 2.8-3.4 GeV |
| Events read, pairs in the fit range | 9,780,165, 402,920 |
| Peak (pole) position, observed | 3.09332 GeV |
| Statistical error, raw | 0.00007 GeV |
| Statistical error, scaled by S = 1.740 | 0.00013 GeV |
| Resolution, observed | 0.02712 ± 0.00012 GeV |
| χ²/ndf **benchmark** | ≤ 2.0 |
| χ²/ndf, observed | 63.6 / 21 = 3.028 (not met) |
| PDG mass (reference only) | 3.0969 GeV |
| Offset from PDG, observed | -0.00358 GeV |
| Fit-choice spread, observed | 0.00006 GeV |

Neither peak position is a mass measurement, because no momentum-scale corrections are applied.

In [ ]:
NOTEBOOK = "03_modular_usage.ipynb"
NOTEBOOK_CHANNELS = ("Z_mumu", "Jpsi_mumu")
NOTEBOOK_FIGURES = ("z_peak_fit.png", "jpsi_peak_fit.png")
# Structural self-checks: every number stated in the "Production results" text equals results/production.json.
def gev(value, channel, sign=False):
    """A GeV value with the channel's decimals, as in the text."""
    decimals = 4 if channel == "Z_mumu" else 5
    if value is None:
        return "n/a"
    return f"{value:+.{decimals}f}" if sign else f"{value:.{decimals}f}"


def stated_numbers(channel):
    """The text's numbers for one channel, recomputed here from the production file."""
    c = PRODUCTION["channels"][channel]
    r = c["result"]
    numbers = {
        "peak position": gev(r["peak_position_gev"], channel),
        "raw stat. error": gev(r["stat_error_gev"], channel),
        "scaled stat. error": gev(r["stat_error_scaled_gev"], channel),
        "scale factor": f"{r['scale_factor']:.3f}",
        "resolution": gev(r["resolution_gev"], channel),
        "resolution error": gev(r["resolution_error_gev"], channel),
        "chi2": f"{r['chi2']:.1f}",
        "ndf": str(r["ndf"]),
        "chi2/ndf": f"{r['chi2_ndf']:.3f}",
        "chi2/ndf benchmark": f"{c['t2']['benchmark']:.1f}",
        "PDG mass": f"{r['pdg_mass_gev']:.4f}",
        "offset": gev(r["offset_from_pdg_gev"], channel, sign=True),
        "fit-choice spread": gev(c["fit_choice_spread_gev"], channel),
        "pairs": f"{r['n_pairs_in_fit_range']:,}",
        "events": f"{r['n_events']:,}",
        "fit range low": f"{r['fit_range_gev'][0]:g}",
        "fit range high": f"{r['fit_range_gev'][1]:g}",
        "files": str(PRODUCTION["sample"]["n_files"]),
        "benchmark files": str(PRODUCTION["targets"]["T4"]["benchmark"]["n_files"]),
    }
    return {f"{channel} {name}": value for name, value in numbers.items()}


notebook = json.loads((ROOT / "notebooks" / NOTEBOOK).read_text())
production_cells = [cell for cell in notebook["cells"] if "production-text" in cell["metadata"].get("tags", [])]
text = "\n".join("".join(cell["source"]) for cell in production_cells)
stated = {}
for channel in NOTEBOOK_CHANNELS:
    stated.update(stated_numbers(channel))
for name, value in stated.items():
    checks.check(f"the text states the {name} of production.json", value.lstrip("+") in text, value)
visible = re.sub(r"\]\([^)]*\)", "]", text)
visible = re.sub(r"<!--.*?-->", "", visible, flags=re.S).replace("Run 2012", "").replace("8 TeV", "")
found = re.findall(r"(?<![\w.])\d[\d,]*(?:\.\d+)?(?![\w])", visible)
known = {value.lstrip("+-") for value in stated.values()}
typed = sorted({n.rstrip(",") for n in found} - known)
checks.check("every number in the text comes from production.json", not typed, ", ".join(typed) or "none other")
for name in NOTEBOOK_FIGURES:
    path = FIGURES / name
    checks.check(f"docs/figures/{name} exists and is listed in production.json",
                 path.exists() and f"docs/figures/{name}" in PRODUCTION["figures"])

In [ ]:
display(Image(filename=str(FIGURES / "z_peak_fit.png"), width=720))
display(Image(filename=str(FIGURES / "jpsi_peak_fit.png"), width=720))
display(Markdown((FIGURES / "jpsi_peak_fit_caption.md").read_text()))

Those are the figures of the two headline fits. Their caption files in `docs/figures/`, one shown here, list every fitted parameter.

In [ ]:
if checks.failures:
    raise AssertionError(f"{len(checks.failures)} check(s) failed: {', '.join(checks.failures)}")
print("All checks passed")